In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/clinical_trial_data.csv')
print("Shape:", df.shape)
df.head()

Shape: (1000, 17)


,patient_id,age,gender,bmi,trial_phase,disease_type,treatment_arm,site_distance_km,visits_completed,visits_missed,adverse_events,protocol_deviations,days_in_trial,employment_status,has_caregiver,insurance_coverage,dropout
0,1,56,Male,26.6,Phase II,Neurology,Drug,122,9,5,1,0,97,Employed,0,1,1
1,2,69,Female,37.2,Phase III,Neurology,Drug,149,7,1,0,1,20,Unemployed,1,0,1
2,3,46,Male,38.7,Phase I,Oncology,Drug,84,1,1,1,2,38,Employed,1,0,1
3,4,32,Female,27.8,Phase II,Cardiology,Placebo,116,8,2,0,1,69,Unemployed,1,0,1
4,5,60,Male,31.1,Phase III,Cardiology,Drug,116,11,3,1,2,70,Unemployed,0,1,1


In [2]:
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing: {df.isnull().sum().sum()}")

Missing values per column:
patient_id             0
age                    0
gender                 0
bmi                    0
trial_phase            0
disease_type           0
treatment_arm          0
site_distance_km       0
visits_completed       0
visits_missed          0
adverse_events         0
protocol_deviations    0
days_in_trial          0
employment_status      0
has_caregiver          0
insurance_coverage     0
dropout                0
dtype: int64

Total missing: 0


In [3]:
print(f"Duplicate rows: {df.duplicated().sum()}")

Duplicate rows: 0


In [4]:
print(df.dtypes)

patient_id               int64
age                      int64
gender                  object
bmi                    float64
trial_phase             object
disease_type            object
treatment_arm           object
site_distance_km         int64
visits_completed         int64
visits_missed            int64
adverse_events           int64
protocol_deviations      int64
days_in_trial            int64
employment_status       object
has_caregiver            int64
insurance_coverage       int64
dropout                  int64
dtype: object


In [ ]:
df['visit_compliance_rate'] = df['visits_completed'] / (
    df['visits_completed'] + df['visits_missed'] + 1
)

df['trial_burden_score'] = df['adverse_events'] + (df['protocol_deviations'] * 2)

df['high_risk_flag'] = (
    (df['site_distance_km'] > 75) & (df['visits_missed'] > 3)
).astype(int)

print("New features created!")
print(df[['visit_compliance_rate', 'trial_burden_score', 'high_risk_flag']].head())

✅ New features created!
   visit_compliance_rate  trial_burden_score  high_risk_flag
0               0.600000                   1               1
1               0.777778                   2               0
2               0.333333                   5               0
3               0.727273                   2               0
4               0.733333                   5               0


In [6]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
print("Categorical columns:", cat_cols)

Categorical columns: ['gender', 'trial_phase', 'disease_type', 'treatment_arm', 'employment_status']


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in cat_cols:
    df[col] = le.fit_transform(df[col])
    print(f"Encoded: {col}")

print("\nAll columns are now numeric:")
print(df.dtypes)

✅ Encoded: gender
✅ Encoded: trial_phase
✅ Encoded: disease_type
✅ Encoded: treatment_arm
✅ Encoded: employment_status

All columns are now numeric:
patient_id                 int64
age                        int64
gender                     int64
bmi                      float64
trial_phase                int64
disease_type               int64
treatment_arm              int64
site_distance_km           int64
visits_completed           int64
visits_missed              int64
adverse_events             int64
protocol_deviations        int64
days_in_trial              int64
employment_status          int64
has_caregiver              int64
insurance_coverage         int64
dropout                    int64
visit_compliance_rate    float64
trial_burden_score         int64
high_risk_flag             int64
dtype: object


In [ ]:
X = df.drop(columns=['patient_id', 'dropout'])
y = df['dropout']

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print(f"\nFeature columns ({len(X.columns)}):")
for col in X.columns:
    print(f"  - {col}")

Features shape: (1000, 18)
Target shape: (1000,)

Feature columns (18):
  - age
  - gender
  - bmi
  - trial_phase
  - disease_type
  - treatment_arm
  - site_distance_km
  - visits_completed
  - visits_missed
  - adverse_events
  - protocol_deviations
  - days_in_trial
  - employment_status
  - has_caregiver
  - insurance_coverage
  - visit_compliance_rate
  - trial_burden_score
  - high_risk_flag


In [9]:
print("Before SMOTE:")
print(y.value_counts())
print(f"Dropout rate: {y.mean()*100:.1f}%")

Before SMOTE:
dropout
1    824
0    176
Name: count, dtype: int64
Dropout rate: 82.4%


In [10]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_balanced, y_balanced = smote.fit_resample(X, y)

print("\nAfter SMOTE:")
print(pd.Series(y_balanced).value_counts())
print(f"New dropout rate: {pd.Series(y_balanced).mean()*100:.1f}%")
print(f"New dataset size: {X_balanced.shape[0]} patients")


After SMOTE:
dropout
1    824
0    824
Name: count, dtype: int64
New dropout rate: 50.0%
New dataset size: 1648 patients


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_balanced)

X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("Features scaled!")
print("\nSample of scaled data:")
print(X_scaled.head())

✅ Features scaled!

Sample of scaled data:
        age    gender       bmi  trial_phase  disease_type  treatment_arm  \
0  0.666176  1.140910 -0.301167     0.168537      0.585279      -0.827691   
1  1.499196 -0.876493  1.401940     1.425322      0.585279      -0.827691   
2  0.025390  1.140910  1.642945    -1.088247      1.506521      -0.827691   
3 -0.871709 -0.876493 -0.108362     0.168537     -1.257204       1.208181   
4  0.922490  1.140910  0.421850     1.425322     -1.257204      -0.827691   

   site_distance_km  visits_completed  visits_missed  adverse_events  \
0          1.299844          0.983294       1.956081       -0.324243   
1          1.929804          0.288559      -0.447813       -1.048297   
2          0.413234         -1.795644      -0.447813       -0.324243   
3          1.159853          0.635926       0.153161       -1.048297   
4          1.159853          1.678028       0.754134       -0.324243   

   protocol_deviations  days_in_trial  employment_status  has

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_balanced,
    test_size=0.2,
    random_state=42,
    stratify=y_balanced  # ensures equal dropout ratio in both splits
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")
print(f"\nTrain dropout rate: {y_train.mean()*100:.1f}%")
print(f"Test dropout rate:  {y_test.mean()*100:.1f}%")

Training samples: 1318
Testing samples:  330

Train dropout rate: 50.0%
Test dropout rate:  50.0%


In [ ]:
import joblib

# Save datasets
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

# Save the scaler (we'll need it later for new patients)
joblib.dump(scaler, '../models/scaler.pkl')

print("All preprocessed data saved!")
print("Scaler saved to models/scaler.pkl")

✅ All preprocessed data saved!
✅ Scaler saved to models/scaler.pkl
